In [2]:
import os
os.listdir(); os.chdir("/aiau010_scratch/azm0269/clover/")

from clover.utils.utils import notebook_line_magic
notebook_line_magic()

In [3]:
def b2_evaluation_scores(baselines, version: str="v1"):
    """Return baseline rows with (template_name, score) MultiIndex columns.

    Read b2_four_baselines_v1 results from the repository outputs directory.
    For each template, average image scores within each seed, then give
    each seed equal weight. BERT uses BERTScore F1. FID uses template-level
    records only, excluding optional prompt-level records.
    """
    import json
    from pathlib import Path

    import numpy as np
    import pandas as pd

    if isinstance(baselines, str):
        baselines = [baselines]
    baselines = list(dict.fromkeys(baselines))
    output_root = Path("/aiau010_scratch/azm0269/clover/outputs")
    score_fields = {
        "fid": "fid",
        "bert": "bert_f1",
        "clip": "clip",
        "imagereward": "imagereward",
    }
    rows = []
    templates = []
    for baseline in baselines:
        folder = output_root / baseline / f"evals/b2/b2_eval_{version}"
        scores = {}
        expected_groups = None
        for metric, field in score_fields.items():
            path = folder / f"{metric}.json"
            records = json.loads(path.read_text())["records"]
            if metric == "fid":
                records = [record for record in records if record["level"] == "template"]
            frame = pd.DataFrame(records)
            if frame.empty or not {"seed", "template_name", field}.issubset(frame.columns):
                raise ValueError(f"Missing template, seed, or score records in {path}")
            if frame[["seed", "template_name"]].isna().any().any() or not np.isfinite(frame[field].to_numpy(dtype=float)).all():
                raise ValueError(f"Missing template/seed or non-finite scores in {path}")
            groups = set(zip(frame["template_name"], frame["seed"]))
            if expected_groups is not None and groups != expected_groups:
                raise ValueError(f"Metric template/seed sets do not match for {baseline}: {path}")
            expected_groups = groups
            per_seed = frame.groupby(["template_name", "seed"], sort=False)[field].mean()
            for template, value in per_seed.groupby(level="template_name", sort=False).mean().items():
                if template not in templates:
                    templates.append(template)
                scores[(template, metric)] = value
        rows.append(scores)

    columns = pd.MultiIndex.from_product(
        [templates, list(score_fields)], names=["template_name", "score"]
    )
    return pd.DataFrame(
        rows, index=pd.Index(baselines, name="baseline"), columns=columns
    )


In [4]:
### Lower FID is good for diversity and quality. It more closely resembles the true distrubution.
## Methods producing higher FID can be suffering from mode collapse as their generation quality starts conforming to narrow distributions.

In [5]:
scores = b2_evaluation_scores(["ddpo", "b2diffurl", "emo_v5", "sd15"], version="v2")
scores.round(2)

template_name object_behavior                         object_attribute        \
score                     fid  bert  clip imagereward              fid  bert   
baseline                                                                       
ddpo                    49.33  0.68  0.39        0.85            83.16  0.52   
b2diffurl               44.41  0.67  0.38        0.74            39.61  0.49   
emo_v5                  33.36  0.64  0.34        0.10            25.62  0.50   
sd15                     0.00  0.64  0.34        0.11             0.00  0.50   

template_name                   positional_relationship              \
score          clip imagereward                     fid  bert  clip   
baseline                                                              
ddpo           0.30       -0.72                   88.94  0.51  0.27   
b2diffurl      0.32       -0.04                   73.57  0.51  0.27   
emo_v5         0.32       -0.08                   52.51  0.51  0.27   
sd15           0.32       -0.09                    0.00  0.51  0.26   

template_name              
score         imagereward  
baseline                   
ddpo                -0.74  
b2diffurl           -0.69  
emo_v5              -0.74  
sd15                -0.74